# Relatório Preliminar de Análise

Este notebook resume os resultados preliminares do projeto **Emendas PIX**, incluindo a exploração dos dados, análise de correlação entre variáveis e clusterização dos municípios. Não abordamos a modelagem multinível nesta etapa.

As fontes dos dados:
- **Densidade demográfica**: IBGE
- **IDHM**: Atlas Brasil
- **PIB per capita**: IBGE
- **Votos válidos**: TSE
- **Emendas PIX**: Transferência de Recursos do Tesouro Nacional e Câmara dos Deputados
- **Taxa de alfabetização**: IBGE


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from kneed import KneeLocator

sns.set(style='whitegrid', context='talk')

base = pd.read_csv('../data/dados_unificados_prefeitos_200k.csv')
base.head()


## Visão geral

In [ ]:
base.info()

In [ ]:
base.isna().sum()

### Observações iniciais

In [ ]:
qtd_unico_candidato = (base['porcentagem_votos_validos_2024'] == 1).sum()
sem_pix = (base['emendas_pix_per_capita_partido_prefeito_eleito'] == 0).sum()
print('Municípios com único candidato:', qtd_unico_candidato)
print('Municípios sem Emendas PIX:', sem_pix)


## Distribuições

In [ ]:
num_cols = ['porcentagem_votos_validos_2024',
             'emendas_pix_per_capita_partido_prefeito_eleito',
             'idhm_2010','alfabetizacao_2010','pib_per_capita_2021','densidade_demografica_2010']
base[num_cols].hist(figsize=(12,8), bins=30)
plt.tight_layout()
plt.show()


## Correlação entre variáveis

In [ ]:
corr = base[num_cols].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlação')
plt.show()


## Clusterização socioeconômica

In [ ]:
features = base[['idhm_2010','alfabetizacao_2010','pib_per_capita_2021','densidade_demografica_2010']].fillna(base.mean())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
ks = range(1,10)
sse = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    sse.append(km.inertia_)
kl = KneeLocator(ks, sse, curve='convex', direction='decreasing')
optimal_k = kl.elbow or 4
km = KMeans(n_clusters=optimal_k, random_state=42)
base['cluster'] = km.fit_predict(X_scaled)
base['cluster'].value_counts()


In [ ]:
sns.pairplot(base, vars=['idhm_2010','alfabetizacao_2010','pib_per_capita_2021','densidade_demografica_2010'], hue='cluster', palette='Set2')
plt.suptitle('Pairplot por cluster', y=1.02)
plt.show()


Este relatório explorou a base consolidada de municípios, investigando distribuições, correlações e agrupamentos socioeconômicos. Essas etapas fornecem insumos para análises mais avançadas no futuro.
